# P4 — read-only response magnitude / support / timing diagnostics

选择 **CPU** 后“全部运行”，无需 GPU、HF_TOKEN、GH_TOKEN 或手动上传数据。
只读取 Drive 中已经完成的 electricity 和 tourism 两轮 discovery。
不重新推理模型、不生成新样本、不读取 B 验证数据、不重新计算 CI 或 gate，
也不改动原报告、原阈值或论文。

**先对账再诊断：** 与此前提交的两份完整报告逐项核对，验证 24 个 full NPZ
及 receipt 的哈希和身份，并复算 DSA/RGR/D1/G 点估计以确认读取的是同一批响应。
任何不一致都会停止；不要删旧结果或关闭完整性检查来绕过。

诊断保留所有 8 cells、每来源 32 个原始 IDs × 3 seeds。
比较原 RGR、全 24 步 L1 幅值比、oracle 活跃区间外的绝对响应占比、
响应重心/峰值偏移和绝对响应分布距离。幅值与位置可以同时有问题；
区间外响应不一定是正确响应的时间平移，也可能是多余响应。

结果是 **post-discovery descriptive diagnostics**，不能把 0/4 改判为成功。
先在 seed 内汇总，再对 seeds 等权。p10/p50/p90 只是描述性分布分位数，不是 CI。
电力一步为小时、旅游一步为月；不要将步数当作相同实际时长。

Inputs (unchanged):
`MyDrive/tsfm-covariate-faithfulness/p4_electricity_discovery_auto_v1/`
and `p4_tourism_discovery_auto_v1/`.
Output: `MyDrive/tsfm-covariate-faithfulness/p4_response_diagnostics_v1/`.
逐 ID 诊断表仅留在 Drive 的 `private/`；最后下载的 ZIP 只含汇总 JSON/CSV。
This notebook embeds code, configs and owner-supplied aggregate reports only.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness')
OUTPUT = PROJECT / 'p4_response_diagnostics_v1'
print('Read-only inputs under:', PROJECT)
print('Separate diagnostic output:', OUTPUT)


In [ ]:
EXPECTED_SOURCE_SHA256 = '40f8f974a43c8a6e4f42c8754a5b217147ae95c31386df60c3e6b5fe1141db50'
CODE_ZIP_BASE64 = '...'
import base64
import hashlib
import io
import os
import subprocess
import sys
import zipfile

blob = base64.b64decode(CODE_ZIP_BASE64, validate=True)
assert hashlib.sha256(blob).hexdigest() == EXPECTED_SOURCE_SHA256
BOOT = Path('/content') / ('p4_response_diagnostic_code_' + EXPECTED_SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), 'Duplicate snapshot member'
    for name in names:
        path = (BOOT / name).resolve()
        assert path.is_relative_to(BOOT.resolve()), 'Unsafe snapshot path'
        content = archive.read(name)
        if path.exists():
            assert path.read_bytes() == content, f'Existing code differs: {name}'
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(content)
    actual = {p.relative_to(BOOT).as_posix() for p in BOOT.rglob('*') if p.is_file()}
    assert actual == set(names), 'Unexpected files in isolated code snapshot'
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy>=2.0,<3', 'PyYAML>=6.0,<7'], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(BOOT / 'src')
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
print('Verified diagnostic code snapshot:', EXPECTED_SOURCE_SHA256)


In [ ]:
# Fresh process: avoid stale notebook imports. Stream the actual error if checks fail.
command = [sys.executable, '-u', '-m', 'covfaith_p4.response_diagnostics',
           '--repo', str(BOOT), '--project', str(PROJECT), '--output', str(OUTPUT)]
with subprocess.Popen(command, cwd=BOOT, env=ENV, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
if returncode:
    raise RuntimeError('Diagnostic stopped. Copy the actual error above; keep original archives.')


In [ ]:
import json

report = json.loads((OUTPUT / 'response_diagnostics_report.json').read_text())
print(json.dumps({key: report[key] for key in (
    'result_status', 'config_hash', 'source_inventory_sha256', 'counts', 'scope')}, indent=2))
for key, value in report['cells'].items():
    selected = ['rgr', 'full_l1_ratio', 'model_outside_support_fraction',
                'oracle_outside_support_fraction', 'relative_active_capture',
                'centroid_shift_steps', 'centroid_abs_error_steps',
                'peak_abs_error_steps', 'absolute_mass_w1_steps', 'near_zero_model_fraction']
    print(json.dumps({'cell': key, 'step_units': value['step_units'],
                      **{name: value['metrics'][name]['point'] for name in selected}}))
print('Post-discovery descriptions only; no new intervals, gate or confirmation claim.')
print('Private per-ID table stays on Drive; send only the summary ZIP below.')


In [ ]:
from google.colab import files

SUMMARY_FILES = ['response_diagnostics_report.json',
                 'complete_eight_cell_response_diagnostics.csv',
                 'per_seed_response_diagnostics.csv', 'artifact_manifest.json']
manifest = json.loads((OUTPUT / 'artifact_manifest.json').read_text())
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in SUMMARY_FILES:
        content = (OUTPUT / name).read_bytes()
        if name != 'artifact_manifest.json':
            assert hashlib.sha256(content).hexdigest() == manifest[name], name
        info = zipfile.ZipInfo(name, date_time=(2026, 9, 30, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, content)
summary_zip = PROJECT / 'p4_response_diagnostics_summary_v1.zip'
zip_content = buffer.getvalue()
if summary_zip.exists():
    assert summary_zip.read_bytes() == zip_content, 'Existing summary ZIP differs; keep it.'
else:
    summary_zip.write_bytes(zip_content)
print('Summary ZIP:', summary_zip)
print('Summary ZIP SHA-256:', hashlib.sha256(zip_content).hexdigest())
files.download(str(summary_zip))
